# 🏟️ Esercizio RAG — FC IBM DreamTeam Chatbot

---

## Obiettivo

In questo esercizio costruirai **da zero** un chatbot RAG che risponde a domande
sul club di calcio **FC IBM DreamTeam**.

La knowledge base è il file `fc_ibm_dreamteam.txt` che contiene il profilo ufficiale
del club: storia, rosa giocatori, statistiche, stadio, contatti e molto altro.

---

## Cosa imparerai

| Step | Concetto | Strumento |
|---|---|---|
| 1 | Leggere e pulire un documento di testo | Python built-in |
| 2 | Suddividere il testo in chunk con overlap | funzione custom |
| 3 | Creare embedding e indicizzare su un vector store | ChromaDB |
| 4 | Fare una similarity search con la query dell'utente | ChromaDB query |
| 5 | Costruire un prompt contestualizzato e chiamare l'LLM | Ollama |
| 6 | Testare la pipeline completa | Python |

---

## Prerequisiti

- Ollama installato e in esecuzione (`ollama serve`)
- Modello scaricato: `ollama pull llama3.2`
- Ambiente virtuale attivato con i pacchetti: `chromadb`, `sentence-transformers`, `ollama`

```bash
pip install chromadb sentence-transformers ollama
```

---

## 🗺️ Schema della pipeline che costruirai

```
fc_ibm_dreamteam.txt
       │
       ▼
  [STEP 1] Lettura file
       │
       ▼
  [STEP 2] Cleaning (rimuovi separatori, normalizza spazi)
       │
       ▼
  [STEP 3] Chunking (dividi in pezzi con overlap)
       │
       ▼
  [STEP 4] Embedding + ChromaDB (indicizza i chunk)
       │
  Query utente ──▶ [STEP 5] Retrieval (top-K chunk simili)
                                │
                           [STEP 6] Generation (Ollama + prompt)
                                │
                            Risposta 🤖
```

---
## ⚙️ STEP 0 — Installazione dipendenze

Esegui la cella qui sotto per installare (o verificare) i pacchetti necessari.

In [1]:
%pip install chromadb sentence-transformers ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 📄 STEP 1 — Leggi il file

La knowledge base si trova nel file `fc_ibm_dreamteam.txt`.

**Cosa devi fare:**
1. Apri il file in lettura con codifica `utf-8`
2. Salva il contenuto nella variabile `raw_text`
3. Stampa la lunghezza totale in caratteri e un'anteprima delle prime 400 caratteri

💡 **Suggerimento:** usa `open(FILE_PATH, "r", encoding="utf-8")` e il metodo `.read()`.

> **Output atteso:**
> ```
> ✅ File letto: 'fc_ibm_dreamteam.txt'
>    Lunghezza totale: XXXX caratteri
> ```

In [2]:
# ---------------------------------------------------------------------------
# STEP 1 — Leggi il file
# ---------------------------------------------------------------------------

FILE_PATH = "fc_ibm_dreamteam.txt"

# TODO: apri il file e leggi il contenuto
raw_text: str = open(FILE_PATH, "r", encoding="utf-8") .read()

print(f"✅ File letto: '{FILE_PATH}'")
print(f"   Lunghezza totale: {len(raw_text)} caratteri")
print(f"   Numero di righe: {raw_text.count(chr(10))}")
print("\n--- Anteprima (prime 400 caratteri) ---")
print(raw_text[:400])

✅ File letto: 'fc_ibm_dreamteam.txt'
   Lunghezza totale: 7440 caratteri
   Numero di righe: 175

--- Anteprima (prime 400 caratteri) ---
PROFILO UFFICIALE - FC IBM DREAMTEAM
Stagione 2024/2025 | Documento interno del club

SEZIONE 1 - STORIA DEL CLUB
------------------------------

FC IBM DreamTeam è stato fondato nel 2001 a Torino da un gruppo di appassionati di calcio
e tecnologia che lavoravano nel settore IT.
Il nome "DreamTeam" nasce dalla volontà dei fondatori di costruire una squadra 


<details>
<summary>💡 Soluzione Step 1 (clicca per espandere)</summary>

```python
with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()
```
</details>

---
## 🧹 STEP 2 — Pulisci il testo (Cleaning)

Il file contiene righe di separazione (`======`, `------`) e spazi ridondanti
che peggiorano la qualità degli embedding.

**Cosa devi fare:** completa la funzione `clean_text(text)` che:
1. Rimuove le righe composte solo da simboli `=` o `-` (3 o più)
2. Rimuove spazi bianchi a inizio/fine di ogni riga
3. Riduce sequenze di 3+ newline consecutive a massimo 2
4. Rimuove spazi multipli interni alla riga

💡 **Suggerimento:** usa il modulo `re` (regular expressions).
- `re.sub(pattern, replacement, text, flags=re.MULTILINE)` per sostituire
- Il pattern `^[=\-]{3,}\s*$` matcha le righe di soli separatori
- Il pattern `\n{3,}` matcha tre o più newline consecutive

> **Output atteso:**
> ```
> ✅ Testo pulito.
>    Riduzione: X.X%
> ```

In [4]:
# ---------------------------------------------------------------------------
# STEP 2 — Cleaning del testo
# ---------------------------------------------------------------------------

import re  # Importa il modulo per cercare e sostituire pattern nel testo.

def clean_text(text: str) -> str:
    """Pulisce il testo grezzo rimuovendo separatori e spazi ridondanti."""
    # 1. Trova le righe composte da almeno 3 '=' o '-' ed eventuali spazi finali.
    #    MULTILINE fa valere ^ e $ all'inizio e alla fine di ogni riga.
    text = re.sub(r"^[=\-]{3,}\s*$", "", text, flags=re.MULTILINE)

    # 2. Divide il testo in righe e rimuove gli spazi ai bordi di ciascuna riga.
    lines = [line.strip() for line in text.splitlines()]
    # Riunisce le righe mantenendo un singolo carattere di newline tra loro.
    text = "\n".join(lines)

    # 3. Sostituisce gruppi di 3 o più newline con esattamente 2 newline.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 4. Riduce sequenze di spazi o tab interni a un singolo spazio.
    text = re.sub(r"[ \t]+", " ", text)

    # Elimina eventuali spazi e newline rimasti all'inizio o alla fine.
    return text.strip()


# Applica la funzione al contenuto letto nel primo step.
cleaned_text = clean_text(raw_text)

# Conferma il completamento e mostra le lunghezze prima e dopo la pulizia.
print("✅ Testo pulito.")
print(f"   Lunghezza originale:      {len(raw_text)} caratteri")
print(f"   Lunghezza dopo cleaning:  {len(cleaned_text)} caratteri")
# Calcola la percentuale di caratteri eliminati durante il cleaning.
print(f"   Riduzione: {(1 - len(cleaned_text)/len(raw_text))*100:.1f}%")
# Mostra i primi 400 caratteri per controllare visivamente il risultato.
print("\n--- Anteprima testo pulito (prime 400 caratteri) ---")
print(cleaned_text[:400])

✅ Testo pulito.
   Lunghezza originale:      7440 caratteri
   Lunghezza dopo cleaning:  7088 caratteri
   Riduzione: 4.7%

--- Anteprima testo pulito (prime 400 caratteri) ---
PROFILO UFFICIALE - FC IBM DREAMTEAM
Stagione 2024/2025 | Documento interno del club

SEZIONE 1 - STORIA DEL CLUB

FC IBM DreamTeam è stato fondato nel 2001 a Torino da un gruppo di appassionati di calcio
e tecnologia che lavoravano nel settore IT.
Il nome "DreamTeam" nasce dalla volontà dei fondatori di costruire una squadra da sogno,
unendo il meglio della tecnologia e dello sport.
La squadra ha


<details>
<summary>💡 Soluzione Step 2 (clicca per espandere)</summary>

```python
def clean_text(text: str) -> str:
    # 1. Rimuove le righe di soli simboli (=, -)
    text = re.sub(r'^[=\-]{3,}\s*$', '', text, flags=re.MULTILINE)

    # 2. Strip di ogni riga
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(lines)

    # 3. Normalizza newline multipli
    text = re.sub(r'\n{3,}', '\n\n', text)

    # 4. Rimuovi spazi multipli
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()
```
</details>

---
## ✂️ STEP 3 — Chunking del testo

Dividiamo il testo in **chunk** (pezzi) di dimensione controllata.

### Perché fare chunking?
- Gli LLM hanno una **context window** limitata: non possiamo passargli tutto il documento
- Chunk più piccoli migliorano la **precisione del retrieval** (si cerca solo il pezzo rilevante)
- L'**overlap** tra chunk evita di spezzare concetti a metà

```
Testo originale:
[..................................................]

Chunking con chunk_size=300, overlap=60:
[--------chunk1--------]
                 [--------chunk2--------]
                                  [--------chunk3--------]
                 |<-- overlap --->|
```

**Cosa devi fare:** completa la funzione `split_into_chunks(text, chunk_size, overlap, separator)`
che:
1. Divide il testo per `separator` (default: `"\n\n"` — paragrafi)
2. Aggrega i paragrafi in chunk finché non si supera `chunk_size` caratteri
3. Quando un chunk è pieno, lo salva e inizia il successivo partendo dagli ultimi `overlap` caratteri
4. Ogni chunk deve essere un dizionario con chiavi `"text"` e `"metadata"` (almeno `source` e `chunk_index`)

💡 **Suggerimento:** la struttura del dizionario chunk è:
```python
{
    "text": "...",
    "metadata": {
        "source": FILE_PATH,
        "chunk_index": 0,
        "char_count": 123
    }
}
```

> **Output atteso:**
> ```
> ✅ Testo diviso in XX chunk.
>    Dimensione media chunk: XXX caratteri
> ```

In [ ]:
# ---------------------------------------------------------------------------
# STEP 3 — Chunking del testo
# ---------------------------------------------------------------------------

def split_into_chunks(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80,
    separator: str = "\n\n"
) -> list[dict]:
    """
    Divide il testo in chunk con overlap.

    Args:
        text:       testo da dividere
        chunk_size: dimensione massima di ogni chunk (caratteri)
        overlap:    caratteri sovrapposti tra chunk consecutivi
        separator:  separatore di paragrafi

    Returns:
        lista di dizionari {'text': str, 'metadata': dict}
    """
    # TODO: dividi il testo in paragrafi usando il separator
    paragraphs = [p.strip() for p in text.split(separator) if p.strip()] 

    chunks = []
    current_chunk = ""
    chunk_index = 0

    current_section = ""
    next_section = ""


    for para in paragraphs:
        # TODO: se aggiungere il paragrafo supera chunk_size e current_chunk non è vuoto:
        #         1. salva current_chunk come nuovo chunk nella lista chunks
     
        #         2. fai partire il nuovo chunk con gli ultimi `overlap` caratteri
        #            + uno spazio + il paragrafo corrente
        #       altrimenti: aggiungi il paragrafo a current_chunk
        if current_chunk and len(current_chunk) + 1 + len(para) > chunk_size:
            chunk_text = current_chunk.strip()
            chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": FILE_PATH,
                    "chunk_index": chunk_index,
                    "char_count": len(chunk_text)
                }
            })
            current_chunk = current_chunk[-overlap:] + " " + para
            chunk_index += 1
        else:
            current_chunk = (current_chunk + " " + para).strip() if current_chunk else para

    # TODO: non dimenticare di salvare l'ultimo chunk rimasto in current_chunk
    if current_chunk.strip():
        chunk_text = current_chunk.strip()
        chunks.append({
            "text": chunk_text,
            "metadata": {
                "source": FILE_PATH,
                "chunk_index": chunk_index,
                "char_count": len(chunk_text)
            }
        })

    return chunks


# Usa chunk_size=400 e overlap=80
chunks = split_into_chunks(cleaned_text, chunk_size=400, overlap=80)

print(f"✅ Testo diviso in {len(chunks)} chunk.")
print(f"   Dimensione media chunk: {sum(c['metadata']['char_count'] for c in chunks) // len(chunks)} caratteri")
print(f"   Dimensione min:  {min(c['metadata']['char_count'] for c in chunks)} caratteri")
print(f"   Dimensione max:  {max(c['metadata']['char_count'] for c in chunks)} caratteri")
print("\n--- Esempio Chunk #0 ---")
print(chunks[0]['text'])
print("\n--- Esempio Chunk #1 ---")
print(chunks[1]['text'])

✅ Testo diviso in 23 chunk.
   Dimensione media chunk: 383 caratteri
   Dimensione min:  106 caratteri
   Dimensione max:  988 caratteri

--- Esempio Chunk #0 ---
PROFILO UFFICIALE - FC IBM DREAMTEAM
Stagione 2024/2025 | Documento interno del club SEZIONE 1 - STORIA DEL CLUB

--- Esempio Chunk #1 ---
TEAM
Stagione 2024/2025 | Documento interno del club SEZIONE 1 - STORIA DEL CLUB FC IBM DreamTeam è stato fondato nel 2001 a Torino da un gruppo di appassionati di calcio
e tecnologia che lavoravano nel settore IT.
Il nome "DreamTeam" nasce dalla volontà dei fondatori di costruire una squadra da sogno,
unendo il meglio della tecnologia e dello sport.
La squadra ha disputato la sua prima partita ufficiale il 14 aprile 2002, pareggiando 1-1
contro il Futsal Piemonte.
Nel 2005 il club si è affiliato alla Federazione Italiana Giuoco Calcio (FIGC).
Nel 2010 il FC IBM DreamTeam ha conquistato il suo primo campionato regionale di Serie D.
Nel 2015 la squadra ha ottenuto la promozione in Serie C, 

<details>
<summary>💡 Soluzione Step 3 (clicca per espandere)</summary>

```python
def split_into_chunks(text, chunk_size=400, overlap=80, separator="\n\n"):
    paragraphs = [p.strip() for p in text.split(separator) if p.strip()]

    chunks = []
    current_chunk = ""
    chunk_index = 0

    for para in paragraphs:
        if len(current_chunk) + len(para) > chunk_size and current_chunk:
            chunks.append({
                "text": current_chunk.strip(),
                "metadata": {
                    "source": FILE_PATH,
                    "chunk_index": chunk_index,
                    "char_count": len(current_chunk.strip())
                }
            })
            current_chunk = current_chunk[-overlap:] + " " + para
            chunk_index += 1
        else:
            current_chunk = (current_chunk + " " + para).strip() if current_chunk else para

    if current_chunk.strip():
        chunks.append({
            "text": current_chunk.strip(),
            "metadata": {
                "source": FILE_PATH,
                "chunk_index": chunk_index,
                "char_count": len(current_chunk.strip())
            }
        })

    return chunks
```
</details>

---
## 🗄️ STEP 4 — Indicizzazione su ChromaDB

Ora creiamo il **vector store**: ChromaDB trasformerà ogni chunk in un vettore numerico
(embedding) e lo salverà in una struttura ottimizzata per la ricerca per similarità.

### Come funziona?

```
"Sasha Petrov ha segnato 22 gol"  ──▶  [0.12, -0.34, 0.87, ...]  (384 dimensioni)
"Chi è il capocannoniere?"        ──▶  [0.11, -0.31, 0.90, ...]  (molto simile! ✅)
"Qual è il prezzo del biglietto?" ──▶  [-0.55, 0.22, -0.10, ...] (molto diverso ❌)
```

**Cosa devi fare:**
1. Crea un client ChromaDB in-memory con `chromadb.Client()`
2. Crea la funzione di embedding con `SentenceTransformerEmbeddingFunction` usando il modello `"paraphrase-multilingual-MiniLM-L12-v2"`
3. Crea (o ottieni) la collection chiamata `"dreamteam_kb"` con distanza coseno
4. Inserisci i documenti nella collection con `collection.add(...)`

💡 **Suggerimento:** per `collection.add()` ti servono tre liste parallele:
- `documents`: lista di stringhe (i testi dei chunk)
- `ids`: lista di ID univoci, es. `["chunk_0", "chunk_1", ...]`
- `metadatas`: lista di dizionari (i metadati dei chunk)

> **Output atteso:**
> ```
> ✅ Collection 'dreamteam_kb' creata.
>    Chunk indicizzati: XX
> ```

In [9]:
# ---------------------------------------------------------------------------
# STEP 4 — Indicizzazione su ChromaDB
# ---------------------------------------------------------------------------

import chromadb
from chromadb.utils import embedding_functions

# TODO 1: crea un client ChromaDB in-memory
client = chromadb.Client()

# TODO 2: crea la funzione di embedding con SentenceTransformerEmbeddingFunction
#         usa il modello 'paraphrase-multilingual-MiniLM-L12-v2'
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2")

# TODO 3: crea (o ottieni) la collection 'dreamteam_kb'
#         imposta la distanza coseno con metadata={"hnsw:space": "cosine"}
collection_name = 'dreamteam_kb'
if any(c.name == collection_name for c in client.list_collections()):
    client.delete_collection(collection_name)  # Se già esiste la cancello

collection = client.create_collection(
    name=collection_name,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
) # impostiamo la distanza dei coseni


# TODO 4: prepara le tre liste per collection.add()
documents = [chunk['text'] for chunk in chunks]    # lista dei testi
ids        = [f"chunk_{i}" for i in range(len(chunks))]   # lista di id univoci tipo "chunk_0", "chunk_1", ...
metadatas  = [chunk['metadata'] for chunk in chunks]  # lista dei metadati

# TODO 5: inserisci i documenti nella collection
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Collection '{collection.name}' creata.")
print(f"   Chunk indicizzati: {collection.count()}")

c:\Users\A1618apulia\OneDrive - ITS Apulia Digital Maker\Desktop\GENERATIVEAI\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (6.0.0.post1)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Collection 'dreamteam_kb' creata.
   Chunk indicizzati: 23


<details>
<summary>💡 Soluzione Step 4 (clicca per espandere)</summary>

```python
import chromadb
from chromadb.utils import embedding_functions

# 1. Client in-memory
client = chromadb.Client()

# 2. Funzione di embedding
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Collection
collection = client.get_or_create_collection(
    name="dreamteam_kb",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# 4. Prepara le liste
documents = [c['text'] for c in chunks]
ids       = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [c['metadata'] for c in chunks]

# 5. Inserisci
collection.add(documents=documents, ids=ids, metadatas=metadatas)
```
</details>

---
## 🔍 STEP 5 — Retrieval (Similarity Search)

Data la query di un utente, troviamo i chunk più **semanticamente simili** nel vector store.

ChromaDB trasforma la query in un embedding e calcola la **distanza coseno** rispetto
a tutti i chunk indicizzati. I chunk con distanza minore (= similarità maggiore) vengono
restituiti come risultato.

**Cosa devi fare:** completa la funzione `retrieve(query, n_results)` che:
1. Chiama `collection.query(query_texts=[query], n_results=n_results, include=["documents", "metadatas", "distances"])`
2. Costruisce e restituisce una lista di dizionari `{"text": ..., "metadata": ..., "similarity_score": ...}`
3. Lo `similarity_score` si calcola come `1 - distanza` (così 1.0 = identico, 0.0 = opposto)

💡 **Suggerimento:** `results['documents'][0]`, `results['metadatas'][0]` e
`results['distances'][0]` sono tre liste parallele che puoi scorrere con `zip()`.

> **Output atteso per la query `"Chi è il capocannoniere della squadra?"`:**
> ```
> [1] Score: 0.XXXX | Testo: ...Petrov...
> [2] Score: 0.XXXX | Testo: ...
> [3] Score: 0.XXXX | Testo: ...
> ```

In [ ]:
# ---------------------------------------------------------------------------
# STEP 5 — Funzione di Retrieval
# ---------------------------------------------------------------------------

def retrieve(query: str, n_results: int = 3) -> list[dict]:
    """
    Recupera i chunk più rilevanti dalla knowledge base.

    Args:
        query:     domanda dell'utente
        n_results: numero di chunk da recuperare

    Returns:
        lista di dict con 'text', 'metadata' e 'similarity_score'
    """
    # TODO: chiama collection.query con query_texts, n_results e include
    results = ???

    # TODO: costruisci la lista di dizionari con testo, metadati e similarity score
    retrieved = []
    for doc, meta, dist in zip(???, ???, ???):
        retrieved.append(???)

    return retrieved


# --- Test del retrieval ---
test_query = "Chi è il capocannoniere della squadra?"
risultati = retrieve(test_query)

print(f"Query: '{test_query}'")
print(f"\nChunk recuperati ({len(risultati)}):")
for i, r in enumerate(risultati, 1):
    print(f"\n  [{i}] Score: {r['similarity_score']:.4f} | Chunk #{r['metadata']['chunk_index']}")
    print(f"       Testo: {r['text'][:180]}...")

<details>
<summary>💡 Soluzione Step 5 (clicca per espandere)</summary>

```python
def retrieve(query: str, n_results: int = 3) -> list[dict]:
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        retrieved.append({
            "text": doc,
            "metadata": meta,
            "similarity_score": round(1 - dist, 4)
        })

    return retrieved
```
</details>

---
## 🤖 STEP 6 — Generation con Ollama

Ora costruiamo il prompt che passa il **contesto recuperato** all'LLM e generiamo la risposta.

### Come funziona il prompt RAG?

```
┌─────────────────────────────────────────────────────────┐
│  SYSTEM: "Sei il chatbot ufficiale di FC IBM DreamTeam. │
│           Rispondi SOLO in base al contesto fornito."   │
├─────────────────────────────────────────────────────────┤
│  USER:   "CONTESTO:                                      │
│           - Petrov ha segnato 22 gol...                 │
│           - La stagione ha 28 partite...                │
│                                                         │
│           DOMANDA: Chi è il capocannoniere?"            │
└─────────────────────────────────────────────────────────┘
                          │
                          ▼
               🤖 "Il capocannoniere è Aleksandr Petrov
                   con 22 gol nella stagione 2023/24."
```

**Cosa devi fare:** completa la funzione `generate_answer(query, retrieved_chunks)` che:
1. Costruisce la stringa `context` unendo i testi dei chunk recuperati (formato: `[Fonte N]\n{testo}`)
2. Definisce un `system_message` che istruisce l'LLM a rispondere solo in base al contesto
3. Definisce un `user_message` con contesto + domanda
4. Chiama `ollama.chat()` con il modello `llama3.2` passando i due messaggi
5. Restituisce `response['message']['content']`

💡 **Suggerimento:** la chiamata a Ollama ha questa struttura:
```python
import ollama
response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message}
    ]
)
```

In [ ]:
# ---------------------------------------------------------------------------
# STEP 6 — Funzione di Generation con Ollama
# ---------------------------------------------------------------------------

import ollama

OLLAMA_MODEL = "llama3.2"  # assicurati di averlo scaricato: ollama pull llama3.2


def generate_answer(query: str, retrieved_chunks: list[dict]) -> str:
    """
    Genera una risposta usando Ollama con il contesto dei chunk recuperati.

    Args:
        query:            domanda dell'utente
        retrieved_chunks: lista di dict con 'text' e 'metadata'

    Returns:
        la risposta generata dall'LLM come stringa
    """
    # Costruisce il contesto con riferimenti alle fonti
    context = "\n\n".join(
        f"[Fonte {i+1}]\n{c['text']}" for i, c in enumerate(retrieved_chunks)
    )

    # System message: definisce il ruolo dell'assistente
    system_message = (
        "Sei il chatbot ufficiale di FC IBM DreamTeam, la squadra di calcio di Torino. "
        "Rispondi alle domande dei tifosi e degli appassionati basandoti ESCLUSIVAMENTE "
        "sulle informazioni fornite nel contesto. "
        "Se l'informazione non è presente nel contesto, rispondi: "
        "'Non ho questa informazione. Contatta il club su info@fcibmdreamteam.it'. "
        "Rispondi sempre in italiano, in modo entusiasta e professionale."
    )

    # User message: contesto + domanda
    user_message = f"""CONTESTO:

{context}

---
DOMANDA: {query}"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ]
    )
    return response["message"]["content"]


print("✅ Funzione generate_answer definita.")

<details>
<summary>💡 Soluzione Step 6 (clicca per espandere)</summary>

```python
import ollama

OLLAMA_MODEL = "llama3.2"

def generate_answer(query: str, retrieved_chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[Fonte {i+1}]\n{c['text']}" for i, c in enumerate(retrieved_chunks)
    )

    system_message = (
        "Sei il chatbot ufficiale di FC IBM DreamTeam, la squadra di calcio di Torino. "
        "Rispondi alle domande dei tifosi e degli appassionati basandoti ESCLUSIVAMENTE "
        "sulle informazioni fornite nel contesto. "
        "Se l'informazione non è presente nel contesto, rispondi: "
        "'Non ho questa informazione. Contatta il club su info@fcibmdreamteam.it'. "
        "Rispondi sempre in italiano, in modo entusiasta e professionale."
    )

    user_message = f"""CONTESTO:

{context}

---
DOMANDA: {query}"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ]
    )
    return response['message']['content']
```
</details>

---
## 🔗 STEP 7 — Pipeline RAG completa

Combina `retrieve()` e `generate_answer()` in un'unica funzione `rag_chatbot(query)`.

**Cosa devi fare:** scrivi la funzione `rag_chatbot(query, n_results, verbose)` che:
1. Chiama `retrieve(query, n_results)` per ottenere i chunk
2. Se `verbose=True`, stampa i chunk recuperati con il loro score
3. Chiama `generate_answer(query, retrieved_chunks)` per generare la risposta
4. Stampa e restituisce la risposta

Poi **testa la pipeline** con almeno 5 domande diverse sul FC IBM DreamTeam.

> Domande suggerite:
> - `"Chi è il capitano della squadra?"`
> - `"Quanti gol ha segnato Petrov nella stagione 2023/24?"`
> - `"Come si chiama lo stadio e quanti posti ha?"`
> - `"Quale modulo usa l'allenatore?"`
> - `"Quanto costa un abbonamento stagionale in curva?"`

In [ ]:
# ---------------------------------------------------------------------------
# STEP 7 — Pipeline RAG completa
# ---------------------------------------------------------------------------

def rag_chatbot(query: str, n_results: int = 5, verbose: bool = True) -> str:
    """
    Pipeline RAG completa: retrieve + generate.

    Args:
        query:     domanda dell'utente
        n_results: chunk da recuperare
        verbose:   se True, mostra i chunk con score

    Returns:
        risposta generata dall'LLM
    """
    print(f"\n{'='*60}")
    print(f"📩 Domanda: {query}")
    print(f"{'='*60}")

    # TODO: step 1 — chiama retrieve()
    retrieved = ???

    # TODO: step 2 — se verbose, stampa i chunk con score
    if verbose:
        ???

    # TODO: step 3 — chiama generate_answer()
    print("\n⏳ Generazione risposta in corso...")
    answer = ???

    print(f"\n🤖 DreamTeam Bot risponde:\n{answer}")
    print(f"{'='*60}\n")
    return answer


# --- Test con 5 domande ---
domande_test = [
    "Chi è il capitano della squadra?",
    "Quanti gol ha segnato Petrov nella stagione 2023/24?",
    "Come si chiama lo stadio e quanti posti ha?",
    "Quale modulo usa l'allenatore?",
    "Quanto costa un abbonamento stagionale in curva?",
]

for domanda in domande_test:
    rag_chatbot(domanda, verbose=False)

<details>
<summary>💡 Soluzione Step 7 (clicca per espandere)</summary>

```python
def rag_chatbot(query: str, n_results: int = 3, verbose: bool = True) -> str:
    print(f"\n{'='*60}")
    print(f"📩 Domanda: {query}")
    print(f"{'='*60}")

    # Step 1: Retrieve
    retrieved = retrieve(query, n_results=n_results)

    # Step 2: Mostra chunk (se verbose)
    if verbose:
        print(f"\n🔍 Chunk recuperati ({len(retrieved)}):")
        for i, r in enumerate(retrieved, 1):
            bar = '█' * int(r['similarity_score'] * 20)
            print(f"   [{i}] Score: {r['similarity_score']:.4f} {bar}")
            print(f"       {r['text'][:100]}...")

    # Step 3: Generate
    print("\n⏳ Generazione risposta in corso...")
    answer = generate_answer(query, retrieved)

    print(f"\n🤖 DreamTeam Bot risponde:\n{answer}")
    print(f"{'='*60}\n")
    return answer
```
</details>

---
## 🌟 BONUS — Sfide aggiuntive

Hai completato la pipeline base? Prova questi esercizi extra per approfondire!

---

### 🏆 BONUS 1 — Filtraggio per soglia di similarità

Modifica la funzione `retrieve()` in modo che scart**i automaticamente** i chunk
con `similarity_score < threshold` (default: `0.3`).
Questo evita di passare all'LLM contesto irrilevante.

```python
def retrieve(query: str, n_results: int = 3, threshold: float = 0.3) -> list[dict]:
    # ... come prima ...
    # filtra i risultati con score >= threshold
    retrieved = [r for r in retrieved if r['similarity_score'] >= threshold]
    return retrieved
```

Testa con una domanda completamente fuori tema, es. `"Qual è la capitale della Francia?"`
e verifica che la risposta sia corretta (nessun chunk rilevante → il bot dice che non lo sa).

In [ ]:
# ---------------------------------------------------------------------------
# BONUS 1 — Retrieve con soglia di similarità
# ---------------------------------------------------------------------------

# TODO: riscrivi retrieve() con il parametro threshold


# Test con domanda fuori tema
rag_chatbot("Qual è la capitale della Francia?", verbose=True)

---
### 🏆 BONUS 2 — Arricchisci la knowledge base

Aggiungi nuovi fatti al file `fc_ibm_dreamteam.txt` (o a una nuova lista in Python)
e re-indicizzali nella collection.

Idee di contenuto da aggiungere:
- Le prossime 3 partite in calendario
- La formazione tipo dell'allenatore (chi gioca dove)
- Un'intervista inventata al capocannoniere Petrov
- Il menu del punto ristoro del Bit Arena 😄

⚠️ **Attenzione:** se usi la stessa collection, devi fare attenzione agli ID duplicati.
Usa `collection.delete(ids=[...])` prima di reinserire, oppure crea una nuova collection.

In [ ]:
# ---------------------------------------------------------------------------
# BONUS 2 — Aggiungi nuovi documenti alla knowledge base
# ---------------------------------------------------------------------------

nuovi_fatti = [
    # TODO: aggiungi qui i tuoi nuovi fatti
    "...",
]

# TODO: inserisci nuovi_fatti nella collection con ID univoci
#       (es. 'extra_0', 'extra_1', ...)


print(f"Chunk totali dopo aggiornamento: {collection.count()}")

---
### 🏆 BONUS 3 — Versione LangChain

Riscrivi l'intera pipeline usando LangChain (come visto nell'Esempio 4 di `rag_lezione.ipynb`).

Componenti da usare:
- `TextLoader` → sostituisce `open()` + `f.read()`
- `RecursiveCharacterTextSplitter` → sostituisce `split_into_chunks()`
- `HuggingFaceEmbeddings` + `Chroma.from_documents()` → sostituisce ChromaDB SDK diretto
- `vectorstore.as_retriever()` → sostituisce `retrieve()`
- `ChatOllama` + `ChatPromptTemplate` + LCEL → sostituisce `generate_answer()`

La pipeline finale deve essere:
```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
risposta = rag_chain.invoke("Chi è il capocannoniere?")
```

In [ ]:
# ---------------------------------------------------------------------------
# BONUS 3 — Pipeline completa con LangChain
# ---------------------------------------------------------------------------

%pip install -q langchain langchain-community langchain-ollama langchain-chroma langchain-huggingface

# TODO: riscrivi la pipeline completa con LangChain
# Suggerimento: segui i passi dell'Esempio 4 in rag_lezione.ipynb


---
## ✅ Riepilogo di ciò che hai costruito

```
 fc_ibm_dreamteam.txt
        │
        ▼
  clean_text()          ← rimuove rumore dal testo
        │
        ▼
  split_into_chunks()   ← divide in pezzi di ~400 char con 80 char di overlap
        │
        ▼
  ChromaDB.add()        ← calcola embedding e indicizza
        │
  query utente ──▶ retrieve()   ← similarity search → top-3 chunk
                       │
                  generate_answer()  ← prompt + Ollama → risposta
                       │
                  rag_chatbot() ← pipeline completa
```

### Concetti chiave che hai applicato

| Concetto | Dove l'hai usato |
|---|---|
| **Embedding** | Step 4 — ChromaDB converte i chunk in vettori numerici |
| **Similarity Search** | Step 5 — distanza coseno tra query e chunk |
| **Chunking** | Step 3 — divisione del testo con overlap |
| **Augmented Generation** | Step 6 — il contesto recuperato guida l'LLM |
| **Grounding** | Step 6 — il system message vincola l'LLM al contesto |

---

## 📚 Vuoi approfondire?

- Torna a `rag_lezione.ipynb` per vedere implementazioni più avanzate
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [Sentence Transformers](https://www.sbert.net/)
- [Ollama Models](https://ollama.ai/library)
- [RAG Paper originale (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)